# 📚 Research Copilot — Sistema RAG sobre Papers Académicos

**Tarea de Prompt Engineering** | Universidad | Feb 2026

Este notebook implementa un sistema **RAG (Retrieval-Augmented Generation)** que permite
hacer preguntas sobre 21 papers académicos de IA aplicada a mercados financieros.

---

## ¿Qué es RAG?

> **RAG = Retrieval-Augmented Generation**
>
> En lugar de depender únicamente del conocimiento preentrenado del LLM, RAG:
> 1. **Recupera** fragmentos relevantes de una base de conocimiento (los papers)
> 2. **Aumenta** el prompt con ese contexto específico
> 3. **Genera** una respuesta fundamentada en las fuentes reales

## Arquitectura del proyecto

```
PDF (21 papers)
      ↓  [pdfplumber]
  Texto extraído
      ↓  [tiktoken]
  Chunks de tokens  ──→  [text-embedding-3-small]  ──→  Vectores
      ↓
  ChromaDB (base vectorial)

Pregunta del usuario
      ↓  [text-embedding-3-small]
  Vector de la pregunta  ──→  ChromaDB (top-5 similares)
      ↓
  Contexto relevante  ──→  Prompt (1 de 4 estrategias)  ──→  GPT-4o
      ↓
  Respuesta + Citas APA
```

## Archivos del proyecto

| Archivo | Propósito |
|---|---|
| `config.py` | Constantes globales (modelos, rutas, tamaños) |
| `prompts.py` | 4 estrategias de prompt engineering |
| `ingest.py` | PDF → chunks → embeddings → ChromaDB |
| `rag.py` | Motor de consulta RAG |
| `app.py` | Interfaz Streamlit (3 pestañas) |

## ⚙️ Paso 1: Instalación de Dependencias

Instalamos las 7 librerías del proyecto:

| Librería | Versión | Uso en el proyecto |
|---|---|---|
| `openai` | ≥1.0 | Embeddings (`text-embedding-3-small`) y generación (`gpt-4o`) |
| `chromadb` | ≥0.4 | Base de datos vectorial local para guardar y buscar chunks |
| `pdfplumber` | ≥0.10 | Extracción de texto de PDFs (más preciso que PyPDF2) |
| `tiktoken` | ≥0.5 | Tokenizador de OpenAI — divide por tokens, no caracteres |
| `python-dotenv` | ≥1.0 | Carga variables de entorno desde `.env` |
| `streamlit` | ≥1.30 | Framework para construir la interfaz web |
| `pandas` | ≥2.0 | Manipulación de datos para el dashboard |

In [ ]:
!pip install openai chromadb pdfplumber tiktoken python-dotenv streamlit pandas -q
print("✅ Todas las dependencias instaladas correctamente")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.

## 🔑 Paso 2: Configurar la API Key de OpenAI

Usamos **Google Colab Secrets** para no exponer la clave en el código.

> **Antes de ejecutar esta celda:**
> 1. Click en el ícono 🔑 del panel izquierdo de Colab
> 2. Agrega un nuevo secreto con nombre: `OPENAI_API_KEY`
> 3. Valor: tu clave que empieza con `sk-...`

Si no configuras Secrets, la celda te pedirá ingresar la clave manualmente.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("✅ API key cargada desde Colab Secrets")
except Exception:
    from getpass import getpass
    os.environ['OPENAI_API_KEY'] = getpass("Ingresa tu OPENAI_API_KEY: ")
    print("✅ API key ingresada manualmente")

# Verificar que no está vacía
key = os.environ.get('OPENAI_API_KEY', '')
if key and not key.startswith('sk-...'):
    print(f"   Clave: {key[:8]}...{key[-4:]} ({len(key)} caracteres)")
else:
    print("⚠️  La clave parece inválida, verifica tu configuración")

Ingresa tu OPENAI_API_KEY: ··········
✅ API key ingresada manualmente
   Clave: sk-proj-...TDEA (164 caracteres)


## 📁 Paso 3: Montar Google Drive y Ubicar los PDFs

Los 21 PDFs deben estar en una carpeta de tu Google Drive junto con los archivos `.py`.

**Estructura esperada en Drive:**
```
Mi unidad/
└── tarea_prompt/
    ├── config.py
    ├── prompts.py
    ├── ingest.py
    ├── rag.py
    ├── app.py
    ├── requirements.txt
    ├── Stock Price Prediction Using LSTM.pdf
    ├── Machine Learning in Stock Price Prediction.pdf
    └── ... (21 papers en total)
```

Después de montar Drive, ajusta `PDF_FOLDER` si tu carpeta tiene otro nombre.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Ajusta esta ruta si tu carpeta tiene otro nombre ──────────────────────────
PDF_FOLDER = '/content/drive/MyDrive/tarea_prompt'

# Verificar PDFs
import os
from pathlib import Path

pdfs = sorted(Path(PDF_FOLDER).glob('*.pdf'))
print(f"✅ Encontrados {len(pdfs)} PDFs en '{PDF_FOLDER}':")
for pdf in pdfs:
    print(f"  📄 {pdf.name}")

if len(pdfs) == 0:
    print("⚠️  No se encontraron PDFs. Revisa la ruta PDF_FOLDER.")

Mounted at /content/drive
✅ Encontrados 21 PDFs en '/content/drive/MyDrive/tarea_prompt':
  📄 A New Equity Investment Strategy with Artificial.pdf
  📄 AI-driven financial crisis prediction Technical frameworks and implementation.pdf
  📄 AI_Powered_Algorithmic_Trading_Among_Ind.pdf
  📄 Advanced Stock Market Prediction Using Long.pdf
  📄 Algorithmic Trading and Ai A Review of Strategies and Market Impact.pdf
  📄 Artificial intelligence and investment management Structure, strategy, and governance.pdf
  📄 Artificial intelligence-based stock market price.pdf
  📄 ESMA_Warning_on_the_use_of_AI_-_EN.pdf
  📄 El impacto de la IA en la gestión de inversiones.pdf
  📄 Forecasting Stock Market Crisis Events Using Machine Learning Methods.pdf
  📄 Machine Learning in Stock Price Prediction.pdf
  📄 Machine learning approach to stock price crash risk.pdf
  📄 Machine learning models predicting returns why most popular.pdf
  📄 Predicting stock market crashes with machine learning A review and methodolog

## 📋 Módulo 1: `config.py` — Configuración Central

`config.py` centraliza **todas las constantes** del proyecto en un solo lugar.
Si quieres cambiar el modelo, el tamaño de chunks, o las rutas, solo lo cambias aquí.

### ¿Por qué centralizar la configuración?
- Evita tener valores hardcodeados distribuidos en múltiples archivos
- Facilita experimentar con diferentes modelos o parámetros
- Un solo lugar para ajustar antes de hacer deploy

### Decisiones de diseño

**`CHUNK_SIZES = [256, 1024]` — ¿Por qué dos tamaños?**
- **256 tokens** → chunks pequeños, recuperación más precisa, menor contexto por chunk
- **1024 tokens** → chunks grandes, más contexto por chunk, recuperación más amplia
- Permitir elegir permite comparar cuál estrategia funciona mejor según la pregunta

**`CHUNK_OVERLAP = 50` — ¿Por qué solapamiento?**
- Evita perder información que cae justo en el borde entre dos chunks
- Los últimos 50 tokens del chunk N se repiten al inicio del chunk N+1

In [ ]:
# ── config.py — contenido completo ────────────────────────────────────────────
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY   = os.getenv('OPENAI_API_KEY', '')
OPENAI_MODEL     = 'gpt-4o'              # LLM para generar respuestas
EMBEDDING_MODEL  = 'text-embedding-3-small'  # Modelo de embeddings (1536 dims)

CHUNK_SIZES      = [256, 1024]           # Dos granularidades de indexación
CHUNK_OVERLAP    = 50                    # Tokens solapados entre chunks adyacentes

CHROMA_PATH      = '/content/chroma_db' # Directorio de ChromaDB en Colab
PDF_FOLDER_PATH  = PDF_FOLDER           # Definido en la celda anterior
CATALOG_PATH     = '/content/catalog.json'

TOP_K            = 5                     # Chunks a recuperar por consulta

# ── Resumen de configuración ───────────────────────────────────────────────────
print("✅ Configuración cargada:")
print(f"   LLM              : {OPENAI_MODEL}")
print(f"   Embeddings       : {EMBEDDING_MODEL}")
print(f"   Chunk sizes      : {CHUNK_SIZES} tokens")
print(f"   Overlap          : {CHUNK_OVERLAP} tokens")
print(f"   Top-K retrieval  : {TOP_K} chunks")
print(f"   ChromaDB path    : {CHROMA_PATH}")

✅ Configuración cargada:
   LLM              : gpt-4o
   Embeddings       : text-embedding-3-small
   Chunk sizes      : [256, 1024] tokens
   Overlap          : 50 tokens
   Top-K retrieval  : 5 chunks
   ChromaDB path    : /content/chroma_db


## 💬 Módulo 2: `prompts.py` — 4 Estrategias de Prompt Engineering

El **prompt engineering** es el arte de diseñar las instrucciones que le damos al LLM.

### ¿Por qué importa la estrategia de prompting?
El mismo modelo GPT-4o puede dar respuestas **muy diferentes** dependiendo de cómo
formulemos el prompt. Una estrategia bien diseñada puede:
- Aumentar la precisión y relevancia de las respuestas
- Forzar un formato de salida específico (ej: JSON)
- Guiar el razonamiento del modelo
- Reducir alucinaciones al anclar el modelo al contexto

### Las 4 estrategias

| # | Nombre | Técnica central |
|---|---|---|
| 1 | **Delimitadores** | Tags XML `<CONTEXT>` / `<QUESTION>` para separar inputs |
| 2 | **JSON Output** | Fuerza salida estructurada con esquema predefinido |
| 3 | **Few-Shot** | 2 ejemplos Q&A guían el estilo de respuesta |
| 4 | **Chain-of-Thought** | El modelo razona paso a paso antes de responder |

Todas reciben los mismos parámetros: `question`, `context_chunks`, `chat_history`
y devuelven una lista de mensajes lista para la API de OpenAI.

### Función auxiliar: `_build_context_text()`

Antes de definir las estrategias, necesitamos una función que formatee los chunks
recuperados de ChromaDB como texto legible para incluir en el prompt.

Cada chunk incluye:
- Su número de orden `[1], [2], ...`
- El título del paper fuente
- Los autores
- El texto del fragmento

In [ ]:
def _build_context_text(context_chunks: list) -> str:
    '''
    Convierte la lista de chunks recuperados de ChromaDB en texto
    formateado y numerado para insertar en el prompt.

    context_chunks: lista de dicts con keys 'text' y 'metadata'
    '''
    parts = []
    for i, chunk in enumerate(context_chunks, 1):
        title   = chunk.get('metadata', {}).get('title', 'Unknown')
        authors = chunk.get('metadata', {}).get('authors', 'Unknown')
        text    = chunk.get('text', '')
        parts.append(f'[{i}] Title: {title}\nAuthors: {authors}\n{text}')

    # Separamos cada chunk con una línea horizontal para claridad visual
    return '\n\n---\n\n'.join(parts)

print("✅ _build_context_text() definida")

# Demo: cómo luce el output
demo_chunks = [
    {'text': 'LSTM networks show strong performance...', 'metadata': {'title': 'Stock Price Prediction Using LSTM', 'authors': 'Zhang et al.'}},
    {'text': 'Random forests provide baseline accuracy...', 'metadata': {'title': 'ML in Finance', 'authors': 'Smith, J.'}},
]
print("\n--- Output de ejemplo ---")
print(_build_context_text(demo_chunks)[:300])

✅ _build_context_text() definida

--- Output de ejemplo ---
[1] Title: Stock Price Prediction Using LSTM
Authors: Zhang et al.
LSTM networks show strong performance...

---

[2] Title: ML in Finance
Authors: Smith, J.
Random forests provide baseline accuracy...


### Estrategia 1 — Delimitadores (`<CONTEXT>` / `<QUESTION>`)

**Técnica:** Usar etiquetas XML-style para delimitar claramente las secciones del prompt.

**¿Por qué funciona?**
Los LLMs son sensibles a la estructura del texto. Al usar delimitadores explícitos:
- El modelo no confunde el contenido de los papers con la pregunta del usuario
- Es fácil para el modelo identificar qué es contexto vs. qué debe responder
- Reduce el riesgo de que el modelo "responda" al contenido del contexto en lugar de a la pregunta

**Estructura del prompt:**
```
[SYSTEM] Eres asistente de investigación. Usa SOLO el contexto dado.

[USER]
<CONTEXT>
  [1] Title: Stock Price Prediction...
  Authors: Zhang et al.
  LSTM networks have shown...
  ---
  [2] Title: ML in Finance...
</CONTEXT>

<QUESTION>
  What are the main ML methods for stock prediction?
</QUESTION>

Answer concisely citing paper titles.
```

In [ ]:
def strategy_delimiter(question: str, context_chunks: list, chat_history: list) -> list:
    '''
    Estrategia 1: Delimitadores XML-style.
    Separa claramente el contexto de la pregunta usando tags.
    '''
    context_text = _build_context_text(context_chunks)

    # Mensaje de sistema: define el rol y restricciones del modelo
    system_msg = (
        'You are a research assistant specializing in AI and financial markets. '
        'Use ONLY the context provided between the delimiters below to answer questions. '
        'Always cite the paper titles you used.'
    )

    # Mensaje de usuario: contexto + pregunta con delimitadores explícitos
    user_content = (
        '<CONTEXT>\n'
        f'{context_text}\n'
        '</CONTEXT>\n\n'
        '<QUESTION>\n'
        f'{question}\n'
        '</QUESTION>\n\n'
        'Answer concisely citing paper titles.'
    )

    # Construimos la lista de mensajes para la API de OpenAI
    # chat_history permite conversaciones multi-turno
    messages = [{'role': 'system', 'content': system_msg}]
    messages.extend(chat_history)
    messages.append({'role': 'user', 'content': user_content})
    return messages

print("✅ Estrategia 1 (Delimitadores) definida")
print(f"   Prompt incluye: system msg + chat_history ({{}}) + user msg con <CONTEXT>/<QUESTION>")

✅ Estrategia 1 (Delimitadores) definida
   Prompt incluye: system msg + chat_history ({}) + user msg con <CONTEXT>/<QUESTION>


### Estrategia 2 — JSON Output (Salida Estructurada)

**Técnica:** Forzar al modelo a responder con un JSON que sigue un esquema predefinido.

**¿Por qué funciona?**
Al especificar el esquema exacto, obtenemos datos estructurados que la aplicación puede:
- Parsear con `json.loads()` para procesar cada campo por separado
- Mostrar de forma organizada (confianza, fuentes, hallazgos)
- Integrar en pipelines automatizados

**Esquema JSON esperado:**
```json
{
  "answer": "Respuesta completa en texto...",
  "sources": ["Paper Title 1", "Paper Title 2"],
  "confidence": 0.85,
  "key_findings": ["hallazgo 1", "hallazgo 2", "hallazgo 3"]
}
```

**Consideración importante:** El campo `confidence` es la estimación del modelo sobre
qué tan bien respaldada está su respuesta por el contexto dado (0 = sin evidencia, 1 = muy respaldado).

In [ ]:
def strategy_json_output(question: str, context_chunks: list, chat_history: list) -> list:
    '''
    Estrategia 2: JSON Output estructurado.
    Fuerza al modelo a responder con un JSON de esquema fijo.
    '''
    context_text = _build_context_text(context_chunks)

    # El system message especifica el esquema JSON con detalle
    # "MUST respond ONLY" es intencional — reduce la tendencia del modelo a agregar texto extra
    system_msg = (
        'You are a research assistant specializing in AI and financial markets. '
        'You MUST respond ONLY with valid JSON — no markdown fences, no extra text. '
        'The JSON must have exactly these keys:\n'
        '  "answer": string with the full answer,\n'
        '  "sources": array of paper title strings cited,\n'
        '  "confidence": float between 0 and 1 indicating your confidence,\n'
        '  "key_findings": array of short strings summarizing key findings.'
    )

    user_content = (
        f'Context:\n{context_text}\n\n'
        f'Question: {question}\n\n'
        'Respond ONLY with valid JSON.'
    )

    messages = [{'role': 'system', 'content': system_msg}]
    messages.extend(chat_history)
    messages.append({'role': 'user', 'content': user_content})
    return messages

print("✅ Estrategia 2 (JSON Output) definida")

✅ Estrategia 2 (JSON Output) definida


### Estrategia 3 — Few-Shot (Ejemplos Previos)

**Técnica:** Proporcionar 2 ejemplos de preguntas y respuestas *antes* de la pregunta real.

**¿Por qué funciona?**
El modelo aprende del patrón de los ejemplos:
- **Estilo**: qué tan detallada debe ser la respuesta
- **Formato**: cómo citar las fuentes, estructura de la respuesta
- **Tono**: académico, técnico, nivel de detalle apropiado

Los ejemplos están **hardcodeados** sobre temas del dominio (ML en finanzas), lo que
asegura que el modelo entienda el contexto del proyecto desde el inicio.

**Estructura del prompt:**
```
[SYSTEM] Eres asistente. Sigue el estilo de los ejemplos.

[USER]
### Example 1
Q: What ML methods are used for stock price prediction?
A: Based on the literature, LSTM networks dominate... (Sources: 'Title1', 'Title2')

### Example 2
Q: How does algorithmic trading impact volatility?
A: Research shows a dual effect... (Sources: 'Title3')

### Your Turn
Context: [chunks recuperados]
Q: [pregunta real del usuario]
A:
```

In [ ]:
def strategy_few_shot(question: str, context_chunks: list, chat_history: list) -> list:
    '''
    Estrategia 3: Few-Shot con 2 ejemplos hardcodeados.
    Los ejemplos cubren temas del dominio para guiar el estilo de respuesta.
    '''
    context_text = _build_context_text(context_chunks)

    system_msg = (
        'You are a research assistant specializing in AI and financial markets. '
        'Use the provided context to answer questions. '
        'Follow the style demonstrated in the examples below.'
    )

    # ── 2 ejemplos hardcodeados sobre el dominio del proyecto ─────────────────
    few_shot_examples = (
        '### Example 1\n'
        'Q: What machine learning methods are most commonly used for stock price prediction?\n'
        'A: Based on the literature, LSTM (Long Short-Term Memory) networks dominate stock '
        'price prediction due to their ability to capture temporal dependencies in time series. '
        'Support Vector Machines (SVM) and Random Forests are also widely used for their '
        'interpretability and robustness against overfitting. More recently, transformer-based '
        'architectures have shown competitive performance. '
        "(Sources: 'Stock Price Prediction Using LSTM', 'Machine Learning in Stock Price Prediction')\n\n"

        '### Example 2\n'
        'Q: How does algorithmic trading impact market volatility?\n'
        'A: Research shows a dual effect: algorithmic trading generally improves market liquidity '
        'and price discovery under normal conditions, but can amplify volatility during stress '
        'events through feedback loops and correlated strategies. Regulatory bodies like ESMA '
        'have issued warnings about systemic risks from AI-driven trading. '
        "(Sources: 'Algorithmic Trading and AI: A Review', 'ESMA Warning on the use of AI')\n\n"
    )

    user_content = (
        f'{few_shot_examples}'
        '### Your Turn\n'
        f'Context:\n{context_text}\n\n'
        f'Q: {question}\n'
        'A:'
    )

    messages = [{'role': 'system', 'content': system_msg}]
    messages.extend(chat_history)
    messages.append({'role': 'user', 'content': user_content})
    return messages

print("✅ Estrategia 3 (Few-Shot) definida")
print("   Incluye 2 ejemplos Q&A hardcodeados sobre ML en finanzas")

✅ Estrategia 3 (Few-Shot) definida
   Incluye 2 ejemplos Q&A hardcodeados sobre ML en finanzas


### Estrategia 4 — Chain-of-Thought (Cadena de Razonamiento)

**Técnica:** Pedir al modelo que razone paso a paso *antes* de dar la respuesta final.

**¿Por qué funciona?**
Los LLMs cometen menos errores cuando se les pide pensar en voz alta:
- Articular el razonamiento reduce alucinaciones
- Identificar acuerdos/desacuerdos entre fuentes da más profundidad
- El proceso de síntesis explícita produce respuestas más matizadas

**Analogía:** Es como pedirle a un estudiante "muestra tu trabajo" en lugar de solo dar la respuesta.

**Los 4 pasos explícitos:**
1. ¿Qué dice cada fuente sobre la pregunta?
2. ¿En qué coinciden y difieren las fuentes?
3. ¿Qué brechas o limitaciones revelan?
4. ¿Cuál es la respuesta sintetizada final?

In [ ]:
def strategy_chain_of_thought(question: str, context_chunks: list, chat_history: list) -> list:
    '''
    Estrategia 4: Chain-of-Thought.
    Pide al modelo que muestre su razonamiento paso a paso antes de concluir.
    '''
    context_text = _build_context_text(context_chunks)

    system_msg = (
        'You are a research assistant specializing in AI and financial markets. '
        'Think step by step before giving your final answer. '
        'Show your reasoning process explicitly.'
    )

    # Los 4 pasos guían el razonamiento del modelo de forma estructurada
    user_content = (
        f'Context:\n{context_text}\n\n'
        f'Question: {question}\n\n'
        'Think step by step:\n'
        f"1. What does each source say about '{question}'?\n"
        '2. What are the agreements and disagreements between sources?\n'
        '3. What gaps or limitations do the sources reveal?\n'
        '4. What is the final synthesized answer, citing relevant papers?\n\n'
        'Work through each step explicitly, then provide your synthesized answer.'
    )

    messages = [{'role': 'system', 'content': system_msg}]
    messages.extend(chat_history)
    messages.append({'role': 'user', 'content': user_content})
    return messages

print("✅ Estrategia 4 (Chain-of-Thought) definida")

✅ Estrategia 4 (Chain-of-Thought) definida


### Dispatcher: `STRATEGIES` y `build_messages()`

El diccionario `STRATEGIES` actúa como un **registro** de todas las estrategias.
La función `build_messages()` es el **router** que despacha al handler correcto.

Este patrón permite agregar nuevas estrategias fácilmente: solo agregar una entrada al diccionario.

In [ ]:
# Registro central de estrategias — patrón Strategy (GoF)
STRATEGIES = {
    1: {'name': 'Delimiters',       'description': 'XML-style <CONTEXT>/<QUESTION> tags',       'fn': strategy_delimiter},
    2: {'name': 'JSON Output',      'description': 'Structured JSON with confidence score',      'fn': strategy_json_output},
    3: {'name': 'Few-Shot',         'description': '2 example Q&A pairs before the question',    'fn': strategy_few_shot},
    4: {'name': 'Chain-of-Thought', 'description': 'Step-by-step reasoning through sources',    'fn': strategy_chain_of_thought},
}


def build_messages(question: str, context_chunks: list, chat_history: list,
                   strategy: int = 1) -> list:
    '''Router: despacha a la función de estrategia correcta.'''
    if strategy not in STRATEGIES:
        raise ValueError(f'Estrategia debe ser 1-4, recibido: {strategy}')
    return STRATEGIES[strategy]['fn'](question, context_chunks, chat_history)


print("✅ Dispatcher build_messages() definido")
print("\n   Estrategias disponibles:")
for k, v in STRATEGIES.items():
    print(f"   {k}. {v['name']:20s} — {v['description']}")

✅ Dispatcher build_messages() definido

   Estrategias disponibles:
   1. Delimiters           — XML-style <CONTEXT>/<QUESTION> tags
   2. JSON Output          — Structured JSON with confidence score
   3. Few-Shot             — 2 example Q&A pairs before the question
   4. Chain-of-Thought     — Step-by-step reasoning through sources


## 📥 Módulo 3: `ingest.py` — Pipeline de Ingesta

Este módulo convierte los PDFs crudos en vectores buscables almacenados en ChromaDB.

### El pipeline completo

```
PDF (archivo)
    │
    ▼  [pdfplumber]
Texto completo + metadata (título, autores, año)
    │
    ├──▶ [tiktoken, size=256,  overlap=50]  ──▶  Lista de chunks pequeños
    │
    └──▶ [tiktoken, size=1024, overlap=50]  ──▶  Lista de chunks grandes
              │                                          │
              ▼  [text-embedding-3-small]               ▼
         Embeddings 256                          Embeddings 1024
              │                                          │
              ▼  [ChromaDB upsert]                       ▼
         Colección papers_256                    Colección papers_1024
```

Al final, se guarda `catalog.json` con estadísticas de cada paper indexado.

### ¿Por qué `upsert` en lugar de `insert`?
`upsert` = update + insert. Si el chunk ya existe (mismo ID), lo actualiza.
Esto permite **re-indexar** los PDFs sin duplicar datos en ChromaDB.

### Función 1: `extract_text_from_pdf()`

Extrae el texto completo del PDF y su metadata usando `pdfplumber`.

**Estrategia para obtener metadata:**
1. **Primero**: lee el campo de metadata del PDF (`pdf.metadata`)
   - `Title` → título del paper
   - `Author` → autores
   - `CreationDate` → año (con regex `\d{4}`)
2. **Fallback**: si no hay año en metadata, busca con regex `\b(19|20)\d{2}\b` en la primera página
3. **Último recurso**: usa el nombre del archivo como título

`pdfplumber` es superior a PyPDF2 porque:
- Maneja mejor los PDFs con columnas múltiples
- Extrae texto con posiciones espaciales
- Funciona mejor con PDFs escaneados con OCR

In [ ]:
import re
import sys
from pathlib import Path
import pdfplumber


def extract_text_from_pdf(path: str) -> dict:
    '''
    Extrae texto y metadata de un PDF.
    Devuelve: {text, title, authors, year, pages}
    '''
    result = {
        'text': '',
        'title': Path(path).stem,   # nombre de archivo como fallback para título
        'authors': 'Unknown',
        'year': None,
        'pages': 0,
    }
    pages_text = []

    try:
        with pdfplumber.open(path) as pdf:
            result['pages'] = len(pdf.pages)

            # ── 1) Intentar metadata embebida en el PDF ──────────────────────
            meta = pdf.metadata or {}
            if meta.get('Title'):
                result['title'] = meta['Title'].strip()
            if meta.get('Author'):
                result['authors'] = meta['Author'].strip()
            if meta.get('CreationDate'):
                year_match = re.search(r'(\d{4})', meta['CreationDate'])
                if year_match:
                    result['year'] = int(year_match.group(1))

            # ── 2) Extraer texto página por página ───────────────────────────
            for page in pdf.pages:
                text = page.extract_text() or ''
                pages_text.append(text)

    except Exception as exc:
        print(f'  [WARN] pdfplumber error en {path}: {exc}', file=sys.stderr)

    result['text'] = '\n'.join(pages_text)

    # ── 3) Fallback: buscar año con regex en la primera página ───────────────
    if result['year'] is None and pages_text:
        year_match = re.search(r'\b(19|20)\d{2}\b', pages_text[0])
        if year_match:
            result['year'] = int(year_match.group())

    if result['year'] is None:
        result['year'] = 0  # año desconocido

    return result


# ── Test rápido con el primer PDF ────────────────────────────────────────────
if pdfs:
    test_pdf = str(pdfs[0])
    print(f"Probando con: {Path(test_pdf).name}")
    info = extract_text_from_pdf(test_pdf)
    print(f"  Título   : {info['title']}")
    print(f"  Autores  : {info['authors']}")
    print(f"  Año      : {info['year']}")
    print(f"  Páginas  : {info['pages']}")
    print(f"  Texto    : {len(info['text'])} caracteres")
    print(f"  Preview  : {info['text'][:200]}...")

Probando con: A New Equity Investment Strategy with Artificial.pdf
  Título   : A New Equity Investment Strategy with Artificial
  Autores  : Unknown
  Año      : 2024
  Páginas  : 15
  Texto    : 38267 caracteres
  Preview  : CIRJE-F- 1230
A New Equity Investment Strategy with Artificial
Intelligence, Multi Factors, and Technical Indicators
Daiya Mita
Nomura Asset Management Co., Ltd.
Akihiko Takahashi
The University of To...


### Función 2: `chunk_text()`

Divide el texto en fragmentos usando **tokens** (no caracteres ni palabras).

**¿Por qué tokens?**
Los modelos de OpenAI procesan tokens. Un token ≈ 0.75 palabras en inglés.
Usar tokens garantiza que cada chunk ocupe exactamente el espacio correcto
en el contexto del modelo.

**Overlap visual:**
```
Tokens: [0 .... 255 | 206 .... 461 | 412 .... 667]
         ←── chunk 1 ──→
                  ←───── chunk 2 ──────→
                               ←───── chunk 3 ──────→
              ↑overlap↑    ↑overlap↑
              (50 tokens)  (50 tokens)
```

**Tokenizador `cl100k_base`:** Es el mismo que usa GPT-4o y los modelos de embeddings de OpenAI.
Usando el mismo tokenizador garantiza conteos de tokens coherentes.

In [ ]:
import tiktoken


def chunk_text(text: str, chunk_size: int, overlap: int) -> list:
    '''
    Divide texto en chunks de `chunk_size` tokens con `overlap` tokens de solapamiento.
    Usa el tokenizador cl100k_base (el mismo que GPT-4o y text-embedding-3-small).
    '''
    enc    = tiktoken.get_encoding('cl100k_base')
    tokens = enc.encode(text)      # texto → lista de IDs de tokens
    chunks = []
    start  = 0

    while start < len(tokens):
        end          = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk_str    = enc.decode(chunk_tokens)  # IDs de tokens → texto

        if chunk_str.strip():
            chunks.append(chunk_str)

        if end >= len(tokens):
            break

        start += chunk_size - overlap  # avanzar dejando el overlap

    return chunks


# ── Demo comparando los dos tamaños ──────────────────────────────────────────
if pdfs:
    sample_text = extract_text_from_pdf(str(pdfs[0]))['text']
    enc         = tiktoken.get_encoding('cl100k_base')
    total_tokens = len(enc.encode(sample_text))

    chunks_256  = chunk_text(sample_text, 256,  50)
    chunks_1024 = chunk_text(sample_text, 1024, 50)

    print(f"Paper: {pdfs[0].name}")
    print(f"Total tokens   : {total_tokens:,}")
    print(f"Chunks (256)   : {len(chunks_256):3d}  ← más granular, búsquedas más precisas")
    print(f"Chunks (1024)  : {len(chunks_1024):3d}  ← menos chunks, más contexto por chunk")
    print(f"\nPrimer chunk (256 tokens):\n{chunks_256[0][:400]}...")

Paper: A New Equity Investment Strategy with Artificial.pdf
Total tokens   : 11,208
Chunks (256)   :  55  ← más granular, búsquedas más precisas
Chunks (1024)  :  12  ← menos chunks, más contexto por chunk

Primer chunk (256 tokens):
CIRJE-F- 1230
A New Equity Investment Strategy with Artificial
Intelligence, Multi Factors, and Technical Indicators
Daiya Mita
Nomura Asset Management Co., Ltd.
Akihiko Takahashi
The University of Tokyo
July 2024
CIRJE Discussion Papers can be downloaded without charge from:
http://www.cirje.e.u-tokyo.ac.jp/research/03research02dp.html
Discussion Papers are a series of manuscripts in their draft ...


### Función 3: `embed_texts()` — Convertir texto en vectores

Los embeddings son representaciones numéricas del significado semántico del texto.

**¿Cómo funciona la búsqueda vectorial?**
```
"LSTM for stock prediction"  ──→  [0.21, -0.13, 0.87, ..., 0.05]  (1536 números)
"LSTM time series forecasting" ──→  [0.22, -0.11, 0.85, ..., 0.06]  ← muy similar
"Spanish cuisine recipes"      ──→  [-0.71, 0.44, -0.32, ..., 0.89]  ← muy diferente
```

La **similitud coseno** mide el ángulo entre dos vectores:
- coseno = 1.0 → vectores idénticos (misma dirección)
- coseno = 0.0 → vectores ortogonales (sin relación)
- coseno = -1.0 → vectores opuestos

**Batching de 100:** La API de OpenAI acepta hasta 2048 textos por request,
pero usamos batches de 100 para ser conservadores con los rate limits.

In [ ]:
import openai


def embed_texts(texts: list) -> list:
    '''
    Convierte una lista de strings en embeddings usando text-embedding-3-small.
    Retorna lista de vectores (cada vector = lista de 1536 floats).
    '''
    client = openai.OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
    embeddings = []
    batch_size = 7  # Reduced batch_size further to ensure total tokens per API call do not exceed 8192 safely

    for i in range(0, len(texts), batch_size):
        batch    = texts[i : i + batch_size]
        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch,
        )
        embeddings.extend([item.embedding for item in response.data])

    return embeddings


# ── Demo: embedding de una frase ──────────────────────────────────────────────
demo_texts = [
    'LSTM networks for stock price prediction',
    'financial crisis prediction using machine learning',
]
demo_embs = embed_texts(demo_texts)
print(f"✅ Embeddings generados")
print(f"   Número de embeddings : {len(demo_embs)}")
print(f"   Dimensión            : {len(demo_embs[0])} (text-embedding-3-small)")
print(f"   Primeros 5 valores   : {[round(x, 4) for x in demo_embs[0][:5]]}")

# Calcular similitud coseno entre las dos frases
import math
def cosine_sim(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na  = math.sqrt(sum(x*x for x in a))
    nb  = math.sqrt(sum(x*x for x in b))
    return dot / (na * nb)

sim = cosine_sim(demo_embs[0], demo_embs[1])
print(f"\n   Similitud coseno entre las 2 frases: {sim:.4f}")
print(f"   (0=sin relación, 1=idénticas)")

✅ Embeddings generados
   Número de embeddings : 2
   Dimensión            : 1536 (text-embedding-3-small)
   Primeros 5 valores   : [-0.016, -0.0722, -0.0159, -0.0255, 0.0116]

   Similitud coseno entre las 2 frases: 0.4316
   (0=sin relación, 1=idénticas)


### Función 4: `get_or_create_collection()` — Base Vectorial ChromaDB

**ChromaDB** es una base de datos vectorial diseñada para búsqueda semántica:
- Guarda los chunks con sus embeddings y metadata
- Permite queries por similitud vectorial en milisegundos
- Es persistente: los datos sobreviven entre ejecuciones
- Open-source y corre localmente (sin necesidad de servicios externos)

**Colecciones creadas:**
- `papers_256` → todos los chunks de 256 tokens
- `papers_1024` → todos los chunks de 1024 tokens

**Chunk ID determinístico (MD5):**
El ID de cada chunk se genera con MD5 del `filename + chunk_size + chunk_index`.
Esto garantiza que el mismo chunk siempre tenga el mismo ID, lo que permite `upsert` idempotente.

In [ ]:
!pip install chromadb
import chromadb
import hashlib
import json


def get_or_create_collection(client, name: str):
    '''Crea o reutiliza una colección ChromaDB con métrica coseno.'''
    return client.get_or_create_collection(
        name=name,
        metadata={'hnsw:space': 'cosine'},  # algoritmo HNSW con similitud coseno
    )


def _make_chunk_id(filename: str, chunk_size: int, chunk_index: int) -> str:
    '''ID estable y único para cada chunk, basado en MD5.'''
    raw = f'{filename}_{chunk_size}_{chunk_index}'
    return hashlib.md5(raw.encode()).hexdigest()


# ── Verificar que ChromaDB funciona ──────────────────────────────────────────
test_client = chromadb.PersistentClient(path=CHROMA_PATH)
test_col    = get_or_create_collection(test_client, 'test_collection')
print(f"✅ ChromaDB inicializado en '{CHROMA_PATH}'")
print(f"   Colección de prueba: {test_col.name}")
print(f"   Items actuales: {test_col.count()}")

# Demo del sistema de IDs
for i in range(3):
    chunk_id = _make_chunk_id('Stock Price Prediction Using LSTM.pdf', 256, i)
    print(f"   ID chunk {i}: {chunk_id}")

✅ ChromaDB inicializado en '/content/chroma_db'
   Colección de prueba: test_collection
   Items actuales: 0
   ID chunk 0: 8aa87f892449a9bdad7c5619a8427bdc
   ID chunk 1: c708fa2f7110fee23ce49b7847f36a82
   ID chunk 2: 75a2a8ee274bba8557bb2b3459c8172b


### Función Principal: `ingest_all_pdfs()`

Orquesta el pipeline completo para todos los PDFs:

```python
Para cada PDF en la carpeta:
  1. extract_text_from_pdf()  →  texto + metadata
  2. chunk_text(size=256)     →  N chunks de 256 tokens
  3. chunk_text(size=1024)    →  M chunks de 1024 tokens
  4. embed_texts(chunks_256)  →  N embeddings
  5. embed_texts(chunks_1024) →  M embeddings
  6. ChromaDB.upsert()        →  guarda en papers_256 y papers_1024

Al final:
  → catalog.json con stats de cada paper
```

**Nota sobre el costo:**
`text-embedding-3-small` cuesta $0.02 por millón de tokens.
21 papers × ~15,000 tokens × 2 tamaños ≈ 630,000 tokens ≈ **$0.013 USD**.

In [ ]:
def ingest_all_pdfs(pdf_folder: str, chroma_path: str, catalog_path: str) -> list:
    '''
    Pipeline completo de ingesta: PDFs → ChromaDB.
    Retorna el catálogo de papers indexados.
    '''
    pdf_files = sorted(Path(pdf_folder).glob('*.pdf'))
    if not pdf_files:
        print(f"No se encontraron PDFs en '{pdf_folder}'")
        return []

    print(f"📂 Procesando {len(pdf_files)} PDFs...")

    # Inicializar ChromaDB (persiste en disco)
    chroma_client = chromadb.PersistentClient(path=chroma_path)
    collections   = {
        cs: get_or_create_collection(chroma_client, f'papers_{cs}')
        for cs in CHUNK_SIZES
    }

    catalog = []

    for pdf_path in pdf_files:
        print(f'\n📄 {pdf_path.name}')
        extracted = extract_text_from_pdf(str(pdf_path))

        if not extracted['text'].strip():
            print(f'  ⚠️  Sin texto extraíble, saltando...')
            continue

        entry = {
            'filename': pdf_path.name,
            'title':    extracted['title'],
            'authors':  extracted['authors'],
            'year':     extracted['year'],
            'pages':    extracted['pages'],
        }

        for chunk_size in CHUNK_SIZES:
            chunks = chunk_text(extracted['text'], chunk_size, CHUNK_OVERLAP)
            print(f'  • Chunks {chunk_size:4d} tokens: {len(chunks):3d}')
            entry[f'chunk_count_{chunk_size}'] = len(chunks)

            if not chunks:
                continue

            ids       = [_make_chunk_id(pdf_path.name, chunk_size, i) for i in range(len(chunks))]
            metadatas = [
                {
                    'filename':    pdf_path.name,
                    'title':       extracted['title'],
                    'authors':     extracted['authors'],
                    'year':        extracted['year'],
                    'chunk_index': i,
                    'chunk_size':  chunk_size,
                    'page_start':  1,
                }
                for i in range(len(chunks))
            ]

            print(f'  → Generando embeddings...')
            embeddings = embed_texts(chunks)

            # upsert: insert o update si ya existe
            collections[chunk_size].upsert(
                ids=ids,
                documents=chunks,
                embeddings=embeddings,
                metadatas=metadatas,
            )

        catalog.append(entry)

    # Guardar catálogo JSON
    with open(catalog_path, 'w', encoding='utf-8') as f:
        json.dump(catalog, f, ensure_ascii=False, indent=2)

    total_256  = sum(p.get('chunk_count_256', 0)  for p in catalog)
    total_1024 = sum(p.get('chunk_count_1024', 0) for p in catalog)
    print(f'\n✅ Ingesta completa')
    print(f'   Papers    : {len(catalog)}')
    print(f'   Chunks 256: {total_256}')
    print(f'   Chunks 1024: {total_1024}')
    print(f'   Catálogo  : {catalog_path}')
    return catalog

print("✅ ingest_all_pdfs() definida")

✅ ingest_all_pdfs() definida


### ▶️ Ejecutar la Ingesta Completa

Esta celda procesa los 21 PDFs y los indexa en ChromaDB.

> ⏱️ **Tiempo estimado:** 5–15 minutos (dependiendo de la API de OpenAI)
>
> 💰 **Costo estimado:** ~$0.01–0.15 USD con `text-embedding-3-small`
>
> 💾 **Solo necesitas ejecutar esto una vez.** Los datos persisten en `/content/chroma_db/`.
> Si reinicias el runtime de Colab, los datos se pierden (están en `/content`).
> Para persistir entre sesiones, cambia `CHROMA_PATH` a una carpeta de Drive.

In [ ]:
# ── Ejecutar la ingesta completa ──────────────────────────────────────────────
catalog = ingest_all_pdfs(
    pdf_folder   = PDF_FOLDER,
    chroma_path  = CHROMA_PATH,
    catalog_path = CATALOG_PATH,
)

# ── Mostrar tabla resumen ─────────────────────────────────────────────────────
if catalog:
    import pandas as pd
    df = pd.DataFrame(catalog)
    display_cols = ['filename', 'year', 'pages', 'chunk_count_256', 'chunk_count_1024']
    display_cols = [c for c in display_cols if c in df.columns]
    print("\n📊 Resumen del catálogo:")
    print(df[display_cols].to_string(index=False))

📂 Procesando 21 PDFs...

📄 A New Equity Investment Strategy with Artificial.pdf
  • Chunks  256 tokens:  55
  → Generando embeddings...
  • Chunks 1024 tokens:  12
  → Generando embeddings...

📄 AI-driven financial crisis prediction Technical frameworks and implementation.pdf
  • Chunks  256 tokens:  30
  → Generando embeddings...
  • Chunks 1024 tokens:   7
  → Generando embeddings...

📄 AI_Powered_Algorithmic_Trading_Among_Ind.pdf
  • Chunks  256 tokens:  36
  → Generando embeddings...
  • Chunks 1024 tokens:   8
  → Generando embeddings...

📄 Advanced Stock Market Prediction Using Long.pdf
  • Chunks  256 tokens:  50
  → Generando embeddings...
  • Chunks 1024 tokens:  11
  → Generando embeddings...

📄 Algorithmic Trading and Ai A Review of Strategies and Market Impact.pdf
  • Chunks  256 tokens:  47
  → Generando embeddings...
  • Chunks 1024 tokens:  10
  → Generando embeddings...

📄 Artificial intelligence and investment management Structure, strategy, and governance.pdf
  ⚠️  Si

## 🔍 Módulo 4: `rag.py` — Motor de Consulta RAG

Este módulo implementa el ciclo RAG completo: **Retrieve → Augment → Generate**.

### ¿Por qué RAG y no solo GPT-4o?

| Problema con GPT-4o solo | Solución con RAG |
|---|---|
| No conoce estos papers específicos | Inyecta los chunks relevantes en el prompt |
| Puede alucinar citas o resultados | Las respuestas están ancladas al texto real |
| Conocimiento con fecha de corte | Funciona con papers recientes |
| Contexto limitado (no puede leer 21 papers) | Solo inyecta los top-5 más relevantes |

### Flujo de `ask()`:
```
Pregunta
  │
  ▼  embed_texts([pregunta])
Vector de la pregunta (1536 dims)
  │
  ▼  collection.query(query_embedding, n=5)
Top-5 chunks más similares (por cosine similarity)
  │
  ▼  build_messages(question, chunks, history, strategy)
Lista de mensajes para OpenAI
  │
  ▼  client.chat.completions.create(model='gpt-4o', messages=...)
Respuesta del LLM
  │
  ▼  generate_apa_citation() para cada chunk único
Citas APA + respuesta + chunks usados
```

In [ ]:
def load_collections(chroma_path: str) -> dict:
    '''
    Abre ChromaDB y retorna dict {256: Collection, 1024: Collection}.
    Si una colección no existe (ingesta no ejecutada), retorna None.
    '''
    client      = chromadb.PersistentClient(path=chroma_path)
    collections = {}
    for size in [256, 1024]:
        try:
            collections[size] = client.get_collection(f'papers_{size}')
            count = collections[size].count()
            print(f"  ✅ papers_{size}: {count} chunks indexados")
        except Exception:
            collections[size] = None
            print(f"  ⚠️  papers_{size}: no encontrada (¿ejecutaste la ingesta?)")
    return collections


def query_chromadb(collection, question_embedding: list, top_k: int = TOP_K) -> list:
    '''
    Busca los top_k chunks más similares al embedding de la pregunta.
    Retorna lista de {text, metadata, distance}.
    '''
    if collection is None:
        return []

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=min(top_k, collection.count() or 1),
        include=['documents', 'metadatas', 'distances'],
    )

    return [
        {
            'text':     doc,
            'metadata': results['metadatas'][0][i],
            'distance': results['distances'][0][i],   # 0=idéntico, 2=opuesto (coseno invertido)
        }
        for i, doc in enumerate(results['documents'][0])
    ]


# Verificar colecciones
print("Verificando colecciones ChromaDB:")
_collections = load_collections(CHROMA_PATH)

Verificando colecciones ChromaDB:
  ✅ papers_256: 992 chunks indexados
  ✅ papers_1024: 216 chunks indexados


In [ ]:
def generate_apa_citation(metadata: dict) -> str:
    '''
    Genera una cita APA simplificada a partir de la metadata del chunk.
    Formato: Autor (Año). Título. Retrieved from nombre_archivo.pdf
    '''
    authors  = metadata.get('authors', 'Unknown Author')
    year     = metadata.get('year') or 'n.d.'
    title    = metadata.get('title', 'Untitled')
    filename = metadata.get('filename', '')
    return f'{authors} ({year}). {title}. Retrieved from {filename}'


# Demo
demo_meta = {
    'authors':  'Zhang, L., Wang, H.',
    'year':     2023,
    'title':    'Stock Price Prediction Using LSTM',
    'filename': 'Stock Price Prediction Using LSTM.pdf'
}
print("✅ generate_apa_citation() definida")
print(f"\nEjemplo APA:")
print(f"  {generate_apa_citation(demo_meta)}")

✅ generate_apa_citation() definida

Ejemplo APA:
  Zhang, L., Wang, H. (2023). Stock Price Prediction Using LSTM. Retrieved from Stock Price Prediction Using LSTM.pdf


### Función Principal: `ask()`

Esta es la función que llama la interfaz Streamlit con cada pregunta del usuario.
Integra todos los módulos en un flujo de una sola llamada.

**Parámetros:**
- `question` — la pregunta del usuario
- `chunk_size` — 256 o 1024 (qué colección ChromaDB usar)
- `strategy` — 1 a 4 (qué estrategia de prompting aplicar)
- `chat_history` — mensajes anteriores para conversación multi-turno

**Retorna un dict con:**
- `answer` — texto de la respuesta de GPT-4o
- `citations` — lista de citas APA (una por paper único)
- `chunks` — los 5 chunks recuperados (para transparencia/debugging)
- `strategy` y `chunk_size` — parámetros usados

In [ ]:
import openai
import tiktoken # Necesario para tokenizar la pregunta

_cached_collections = {}   # caché para no re-abrir ChromaDB en cada query


def ask(question: str, chunk_size: int = 256, strategy: int = 1,
        chat_history: list = None) -> dict:
    '''
    Función principal del RAG.
    Embed pregunta → retrieve chunks → prompt → GPT-4o → respuesta + citas APA.
    '''
    if chat_history is None:
        chat_history = []

    client = openai.OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

    # Definir el tokenizador y el límite de tokens para el embedding de la pregunta
    enc = tiktoken.encoding_for_model(EMBEDDING_MODEL)
    MAX_EMBEDDING_TOKENS = 8192 # Límite de tokens para text-embedding-3-small

    # ── 1) Embed la pregunta ─────────────────────────────────────────────────
    # Salvaguarda: Asegurar que la pregunta no exceda el límite de tokens
    question_tokens = enc.encode(question)
    if len(question_tokens) > MAX_EMBEDDING_TOKENS:
        # Truncar la pregunta si es demasiado larga
        truncated_question = enc.decode(question_tokens[:MAX_EMBEDDING_TOKENS])
        print(f"⚠️  Pregunta truncada de {len(question_tokens)} a {MAX_EMBEDDING_TOKENS} tokens para embedding.")
    else:
        truncated_question = question

    embed_resp         = client.embeddings.create(model=EMBEDDING_MODEL, input=[truncated_question])
    question_embedding = embed_resp.data[0].embedding

    # ── 2) Cargar colecciones con caché ──────────────────────────────────────
    # Solo usar caché si hay al menos una colección real (no None)
    if not _cached_collections or all(v is None for v in _cached_collections.values()):
        _cached_collections.clear()
        _cached_collections.update(load_collections(CHROMA_PATH))

    collection = _cached_collections.get(chunk_size)
    chunks     = query_chromadb(collection, question_embedding, top_k=TOP_K)

    if not chunks:
        return {
            'answer':     'No hay contenido indexado. Ejecuta la celda de ingesta primero.',
            'citations':  [],
            'chunks':     [],
            'strategy':   strategy,
            'chunk_size': chunk_size,
        }

    # ── 3) Construir prompt con la estrategia elegida ────────────────────────
    messages = build_messages(question, chunks, chat_history, strategy=strategy)

    # ── 4) Llamar a GPT-4o ──────────────────────────────────────────────────
    response    = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages,
        temperature=0.2,    # baja temperatura → más determinista y factual
        max_tokens=1500,
    )
    answer_text = response.choices[0].message.content.strip()

    # ── 5) Citas APA (una por paper, sin duplicados) ─────────────────────────
    seen, citations = set(), []
    for chunk in chunks:
        fname = chunk['metadata'].get('filename', '')
        if fname not in seen:
            seen.add(fname)
            citations.append(generate_apa_citation(chunk['metadata']))

    return {
        'answer':     answer_text,
        'citations':  citations,
        'chunks':     chunks,
        'strategy':   strategy,
        'chunk_size': chunk_size,
    }

print("✅ ask() definida — motor RAG completo listo")

✅ ask() definida — motor RAG completo listo


## 🧪 Pruebas del Sistema — Comparando las 4 Estrategias

Ahora probamos el sistema con preguntas reales.
Compara cómo varía la respuesta según la estrategia.

> **Nota:** Estas celdas requieren que la ingesta haya sido ejecutada exitosamente.

In [ ]:
# ── Prueba 1: Estrategia 1 (Delimitadores) ────────────────────────────────────
print("=" * 65)
print("ESTRATEGIA 1 — DELIMITADORES")
print("Pregunta: What are the main ML methods for stock price prediction?")
print("=" * 65)

result1 = ask(
    question    = 'What are the main ML methods used for stock price prediction?',
    chunk_size  = 256,
    strategy    = 1,
)

print("\n📝 RESPUESTA:")
print(result1['answer'])

print("\n📎 FUENTES RECUPERADAS:")
for cite in result1['citations']:
    print(f"  • {cite}")

print(f"\n🔍 CHUNKS USADOS ({len(result1['chunks'])}):")
for c in result1['chunks']:
    meta = c['metadata']
    print(f"  [{meta.get('title', '?')[:50]}] dist={c['distance']:.4f}")

ESTRATEGIA 1 — DELIMITADORES
Pregunta: What are the main ML methods for stock price prediction?
  ✅ papers_256: 992 chunks indexados
  ✅ papers_1024: 216 chunks indexados

📝 RESPUESTA:
The main machine learning methods used for stock price prediction include:

1. **Regression Models**: These are commonly used for detecting patterns and trends in stock data. (Referenced in "Stock Prediction Using Machine Learning" by Dayanidhi S)

2. **Support Vector Machines (SVM)**: SVMs are used for their ability to handle classification and regression tasks effectively. (Referenced in "Stock Closing Price Prediction using Machine Learning Techniques" by Mehar Vijh and "Artificial intelligence-based stock market price prediction, a review" by Vishwas H N, Usha B A, Sangeetha K N)

3. **Decision Trees**: These are used for their simplicity and interpretability in making predictions. (Referenced in "Stock Prediction Using Machine Learning" by Dayanidhi S)

4. **Ensemble Methods**: These methods, which 

In [ ]:
# ── Prueba 2: Estrategia 2 (JSON Output) ─────────────────────────────────────
import json as json_lib

print("=" * 65)
print("ESTRATEGIA 2 — JSON OUTPUT")
print("Pregunta: What are the limitations of LSTM for financial forecasting?")
print("=" * 65)

result2 = ask(
    question    = 'What are the limitations of LSTM for financial forecasting?',
    chunk_size  = 256,
    strategy    = 2,
)

print("\n📝 RESPUESTA RAW:")
print(result2['answer'])

# Intentar parsear el JSON
print("\n📊 JSON PARSEADO:")
try:
    parsed = json_lib.loads(result2['answer'])
    print(f"  Confianza    : {parsed.get('confidence', 'N/A')}")
    print(f"  Fuentes      : {parsed.get('sources', [])}")
    print(f"  Respuesta    : {str(parsed.get('answer', ''))[:200]}...")
    print(f"  Hallazgos    :")
    for finding in parsed.get('key_findings', []):
        print(f"    - {finding}")
except json_lib.JSONDecodeError as e:
    print(f"  ⚠️  No es JSON válido: {e}")
    print("  (El modelo puede haber añadido texto extra fuera del JSON)")

ESTRATEGIA 2 — JSON OUTPUT
Pregunta: What are the limitations of LSTM for financial forecasting?

📝 RESPUESTA RAW:
{
  "answer": "LSTM models for financial forecasting face several limitations. Firstly, their performance deteriorates in extremely volatile market conditions where external economic and geopolitical factors disrupt historical trends. Secondly, they are sensitive to data quality, meaning the accuracy of forecasts heavily depends on the availability and accuracy of historical stock data. Additionally, LSTMs are prone to overfitting, where the model overemphasizes past trends, reducing its ability to accurately predict future stock prices. Despite these challenges, LSTMs still offer significant improvements over traditional methods, and hybrid approaches incorporating attention mechanisms or macroeconomic indicators are suggested to enhance their predictive robustness.",
  "sources": [
    "Stock Prediction Using Machine Learning",
    "Advanced Stock Market Prediction Using

In [ ]:
# ── Prueba 3: Estrategia 4 (Chain-of-Thought) ─────────────────────────────────
print("=" * 65)
print("ESTRATEGIA 4 — CHAIN-OF-THOUGHT")
print("Pregunta: How is AI used for financial crisis prediction?")
print("=" * 65)

result4 = ask(
    question    = 'How is AI used for financial crisis prediction?',
    chunk_size  = 1024,    # chunks más grandes para más contexto
    strategy    = 4,
)

print("\n📝 RESPUESTA (con razonamiento explícito):")
print(result4['answer'])

print("\n📎 FUENTES:")
for cite in result4['citations']:
    print(f"  • {cite}")

ESTRATEGIA 4 — CHAIN-OF-THOUGHT
Pregunta: How is AI used for financial crisis prediction?

📝 RESPUESTA (con razonamiento explícito):
To understand how AI is used for financial crisis prediction, let's analyze the information provided in the sources step by step:

1. **Source Analysis:**
   - **Source [1]:** AI is used to analyze diverse data streams such as market trends, economic indicators, and geopolitical factors to identify emerging risks with precision. The integration of AI into financial systems enhances efficiency, improves decision-making, and aids in fraud management. The paper highlights the importance of combining AI with human oversight due to challenges like human behavior and unexpected global events.
   - **Source [2]:** The article emphasizes the continuous improvement of AI systems in predicting financial crises, highlighting the importance of sustained performance and the integration of AI with existing systems for effective risk management.
   - **Source [3]:** AI-

In [ ]:
# ── Prueba 4: Conversación multi-turno con Estrategia 3 ───────────────────────
print("=" * 65)
print("ESTRATEGIA 3 — FEW-SHOT (conversación multi-turno)")
print("=" * 65)

# Primera pregunta
r_turn1 = ask(
    question    = 'What role does LSTM play in stock prediction?',
    chunk_size  = 256,
    strategy    = 3,
)
print("\n🙋 Turno 1:")
print(r_turn1['answer'][:400])

# Segunda pregunta con historial
history = [
    {'role': 'user',      'content': 'What role does LSTM play in stock prediction?'},
    {'role': 'assistant', 'content': r_turn1['answer']},
]
r_turn2 = ask(
    question     = 'What are the main alternatives to LSTM mentioned in the papers?',
    chunk_size   = 256,
    strategy     = 3,
    chat_history = history,
)
print("\n🙋 Turno 2 (con contexto del turno anterior):")
print(r_turn2['answer'][:400])

ESTRATEGIA 3 — FEW-SHOT (conversación multi-turno)

🙋 Turno 1:
LSTM (Long Short-Term Memory) networks play a crucial role in stock prediction by effectively capturing long-term temporal dependencies in stock price data. They are particularly well-suited for this task due to their ability to learn from historical data patterns and focus on significant trends while ignoring noise. LSTMs outperform traditional models like RNNs and ANNs in stock price prediction,

🙋 Turno 2 (con contexto del turno anterior):
The main alternatives to LSTM mentioned in the papers include:

1. **Self-Attention Mechanisms and Transformer-based Models**: These models, such as SpectralGPT, are noted for their ability to handle high-dimensional financial data and effectively manage noisy and nonlinear data. They are implemented using libraries like Hugging Face. (Source: [2])

2. **Hybrid Models**: These combine multiple met


## 🖥️ Módulo 5: `app.py` — Interfaz Streamlit

`app.py` construye la interfaz web con **Streamlit**, un framework que convierte
scripts Python en aplicaciones web interactivas sin necesidad de HTML/CSS/JavaScript.

### Estructura de la UI

```
┌─────────────────────────────────────────────────────────┐
│ SIDEBAR                    │  CONTENIDO PRINCIPAL        │
│                            │                             │
│ 📚 Research Copilot        │  [Tab 1] [Tab 2] [Tab 3]    │
│                            │                             │
│ Chunk size: [256 ▼]        │  💬 Chat                    │
│                            │  ─────────────────────────  │
│ Estrategia: [1 ▼]          │  🤖 Respuesta anterior...   │
│  Desc. estrategia          │                             │
│                            │  [Nueva conversación]       │
│ [🔄 Re-indexar PDFs]       │                             │
│                            │  > Escribe tu pregunta...   │
│ Papers: 21                 │                             │
│ Chunks 256: 847            └─────────────────────────────┘
│ Chunks 1024: 267
└────────────────────────────
```

### Las 3 pestañas

**Tab 1 — 💬 Chat:**
- `st.chat_message` / `st.chat_input` para interfaz conversacional
- `st.session_state.messages` para persistir el historial
- Expanders "Ver fuentes" y "Chunks recuperados" para transparencia

**Tab 2 — 📄 Papers:**
- `st.dataframe` con el catálogo completo
- Buscador de texto libre + filtro por año (slider)
- Preview del primer chunk al seleccionar un paper

**Tab 3 — 📊 Dashboard:**
- `st.metric` para KPIs (total papers, chunks)
- `st.bar_chart` para distribución por año
- Tabla y gráfica de uso de estrategias en la sesión actual

In [ ]:
# Copiar app.py y archivos .py desde Drive a /content/
import shutil

files_to_copy = ['app.py', 'config.py', 'prompts.py', 'rag.py', 'ingest.py']

for fname in files_to_copy:
    src = Path(PDF_FOLDER) / fname
    dst = Path('/content') / fname

    if src.exists():
        shutil.copy(str(src), str(dst))
        print(f"✅ {fname} copiado a /content/")
    else:
        print(f"⚠️  {fname} no encontrado en {PDF_FOLDER}")

# Verificar que catalog.json está accesible
cat_src = Path(PDF_FOLDER) / 'catalog.json'
if cat_src.exists():
    shutil.copy(str(cat_src), '/content/catalog.json')
    print("✅ catalog.json copiado a /content/")
elif Path('/content/catalog.json').exists():
    print("✅ catalog.json ya existe en /content/")
else:
    print("⚠️  catalog.json no encontrado — ejecuta la ingesta primero")

✅ app.py copiado a /content/
✅ config.py copiado a /content/
✅ prompts.py copiado a /content/
✅ rag.py copiado a /content/
✅ ingest.py copiado a /content/
✅ catalog.json ya existe en /content/


## 🚀 Lanzar la Aplicación Streamlit

Usamos el **proxy de Google Colab** para exponer el puerto 8501 de forma segura.

**Instrucciones:**
1. Ejecuta la celda
2. Espera ~10 segundos hasta que aparezca la URL
3. Haz clic en la URL para abrir la aplicación

> No se requiere contraseña ni servicios externos.

In [ ]:
import subprocess, time, sys
from google.colab.output import eval_js

# Lanzar Streamlit en background
print("Iniciando Streamlit en puerto 8501...")
streamlit_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', '/content/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    cwd='/content',
)

# Esperar que Streamlit arranque completamente
time.sleep(8)
print("✅ Streamlit iniciado")

# Obtener URL pública a través del proxy de Colab (sin npm, sin servicios externos)
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"\n🌐 Abre esta URL en tu navegador:")
print(url)

Iniciando Streamlit en puerto 8501...
✅ Streamlit iniciado

🌐 Abre esta URL en tu navegador:
https://8501-m-s-1gxvi3vkmrp8s-a.us-east4-0.prod.colab.dev


---

## 📝 Resumen del Proyecto

### Stack tecnológico

| Capa | Tecnología | Función |
|---|---|---|
| Extracción | `pdfplumber` | Leer texto de los PDFs académicos |
| Tokenización | `tiktoken` | Dividir por tokens con overlap |
| Embeddings | `text-embedding-3-small` | Representación vectorial del texto |
| Vector DB | `ChromaDB` | Almacenamiento y búsqueda semántica |
| LLM | `GPT-4o` | Generación de respuestas con contexto |
| Prompting | 4 estrategias | Diferentes técnicas de prompt engineering |
| UI | `Streamlit` | Interfaz conversacional web |

### Las 4 estrategias de Prompt Engineering

| # | Estrategia | Ventaja principal | Caso de uso ideal |
|---|---|---|---|
| 1 | **Delimitadores** | Simple, claro, confiable | Preguntas generales |
| 2 | **JSON Output** | Respuesta estructurada/parseable | Integración con apps |
| 3 | **Few-Shot** | Controla estilo y formato | Respuestas consistentes |
| 4 | **Chain-of-Thought** | Razonamiento profundo | Preguntas complejas/comparativas |

### Parámetros comparables

- **Chunk size 256** vs **1024**: granularidad de recuperación
- **Estrategia 1-4**: técnica de prompting
- Todas las combinaciones trabajan sobre los mismos 21 papers

### Preguntas de prueba sugeridas

- *"What are the main ML methods used for stock price prediction?"*
- *"What are the limitations of LSTM for financial forecasting?"*
- *"How is AI used for financial crisis prediction?"*
- *"What does ESMA say about AI risks in financial markets?"*
- *"Compare the performance of different ML models in the reviewed papers"*

In [ ]:
# 1. Instalamos la herramienta de conversión
!pip install nbconvert

# 2. Convertimos el archivo (reemplaza 'tu_archivo.ipynb' por el nombre real)
# Nota: En Colab, el archivo suele llamarse 'colab_research_copilot.ipynb'
!jupyter nbconvert --to markdown "colab_research_copilot (1).ipynb"

[NbConvertApp] WARNING | pattern 'colab_research_copilot (1).ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True